In [ ]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 04, Lie algebroid

Companion notebook to [04_lie_algebroid.md](04_lie_algebroid.md). The `(E, [·,·]_E, ρ)` triple as a single object, anchor compatibility as a separate axiom, and the algebroid Cartan bundle.

## The triple `(E, [·,·]_E, ρ)`

`LieAlgebroid` keeps the bundle name, the section bracket, the anchor, and the target `TM` bracket together in one object.

In [ ]:
from jacopy import VectorFields
from jacopy.brackets.lie import LieBracket
from jacopy.calculus.anchor import Anchor
from jacopy.core.expr import Symbol
from jacopy.core.registry import PropertyRegistry
from jacopy.library.lie_algebroid import LieAlgebroid

reg = PropertyRegistry()
E = Symbol("E")
bracket_E = LieBracket(name="[·,·]_E")
rho = Anchor(name="ρ")

A = LieAlgebroid(E, bracket=bracket_E, anchor=rho, name="E-algebroid")
print(A)

## Anchor compatibility, a separate axiom

`ρ([X, Y]_E) = [ρ(X), ρ(Y)]_{TM}` is part of the Lie algebroid **definition**; the bracket's own axioms do not entail it. Three presentations: an obstruction (Expr), a condition (VanishingCondition), and a single-step `axiom`-tagged `ProofChain`.

In [ ]:
X, Y = VectorFields("X Y", registry=reg)

print('obstruction:')
print(' ', A.anchor_compatibility_obstruction(X, Y, reg))
print()
print('condition:', A.anchor_compatibility_condition(X, Y, reg))
print()
chain = A.prove_anchor_compatibility(X, Y, registry=reg)
print('chain rule:', chain.steps[0].rule)
print('provenance:', chain.steps[0].provenance_tag)
print('justification:', chain.steps[0].justification)

## Algebroid Cartan bundle

Same `CartanCalculus` API but with `E`-tagged operators: `d_E`, `L_{E,X}`, `ι_{E,X}`. The operator-level `relation()` call returns an `OperatorEquation`. Note: algebroid Cartan magic does not auto-close under `verify` (the engine's Cartan-definition rewrite is bound to TM); this is expected and recorded. Live proofs of the five Cartan relations on TM live in [05_cartan_calculus.md](05_cartan_calculus.md).

In [ ]:
cart = A.cartan
print('d:', A.d)
print('L_E,X:', cart.lie_derivative(X))
print('ι_E,X:', cart.interior(X))

eq = cart.relation("cartan_magic", X=X)
print('magic:', eq.lhs, '=', eq.rhs)

## Seeded theorem

The compatibility axiom is registered in `theorem_book`, downstream theorems (algebroid Cartan, Courant–Dorfman bridge) cite it in a single step.

In [ ]:
from jacopy.library import theorem_book

thm = theorem_book.get("lie_algebroid_anchor_compat")
print('statement:', thm.statement)
print('from_axioms:', thm.from_axioms)

## Next step

Live proofs of the five Cartan relations on TM, in two modes: [05_cartan_calculus.md](05_cartan_calculus.md).